## 1. Подготовка пайпланов для обработки признаков

In [51]:
import os
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
from sklearn.metrics import recall_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

from sklearn.model_selection import (cross_val_score, train_test_split, PredefinedSplit,
                                     GridSearchCV, StratifiedKFold, RandomizedSearchCV,
                                     cross_val_predict)


In [3]:
load_path = os.path.join('..', 'data', 'processed', 'eda_telco_customer_churn_train.csv')

df = pd.read_csv(load_path)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4225 entries, 0 to 4224
Data columns (total 24 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Age                                4225 non-null   int64  
 1   Avg Monthly GB Download            4225 non-null   int64  
 2   Avg Monthly Long Distance Charges  4225 non-null   float64
 3   CLTV                               4225 non-null   int64  
 4   Contract                           4225 non-null   object 
 5   Device Protection Plan             4225 non-null   int64  
 6   Gender                             4225 non-null   object 
 7   Internet Service                   4225 non-null   int64  
 8   Multiple Lines                     4225 non-null   int64  
 9   Number of Dependents               4225 non-null   int64  
 10  Offer                              4225 non-null   int64  
 11  Online Backup                      4225 non-null   int64

In [4]:
y_train = df['Churn']
X_train = df.drop(columns=['Churn'])

In [5]:
numeric = ['Age', 'Avg Monthly GB Download', 'Avg Monthly Long Distance Charges', 
           'CLTV', 'Satisfaction Score', 'Tenure in Months', 'Total Long Distance Charges']
print(f'Всего {len(numeric)} числовых признаков')

Всего 7 числовых признаков


In [6]:
categorical = ['Contract', 'Device Protection Plan', 'Gender', 'Internet Service', 'Multiple Lines', 'Offer', 
              'Online Backup', 'Online Security', 'Paperless Billing', 'Partner',
              'Payment Method', 'Premium Tech Support', 'Streaming TV', 'Total Refunds',
              'Number of Dependents', 'Total Extra Data Charges']
print(f'Всего {len(categorical)} категориальных признаков')

Всего 16 категориальных признаков


In [7]:
num_transformer = Pipeline([
    ('scaler', StandardScaler())
])
cat_transformer = Pipeline([
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, numeric),
        ('cat', cat_transformer, categorical)
    ])

Используем StratifiedKFold для кросс-валидации, который позволит сохранять баланс классов в каждом фолде. А валидационный датасет будем исопльзовать для итоговой оценки модели.

In [63]:
load_path = os.path.join('..', 'data', 'processed', 'eda_telco_customer_churn_val.csv')

df_val = pd.read_csv(load_path)
X_val = df_val.drop(columns=['Churn'])
y_val = df_val['Churn']

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## 2. Логистическая регрессия

In [ ]:
lr_model = Pipeline([
    ("preprocessor", preprocessor),
    ("logreg", LogisticRegression(random_state=42))
])

lr_param_grid = {
    'logreg__C': [0.01, 0.1, 1, 10, 100],
    'logreg__l1_ratio': [0, 0.5, 1],
    'logreg__solver': ['saga'],
    'logreg__max_iter': [1000, 2000, 3000]
}

lr_cv = GridSearchCV(
    lr_model,
    param_grid=lr_param_grid,
    cv=cv,
    scoring='recall',
    verbose=1)

lr_cv.fit(X_train, y_train)
print('Baseline (Classification):')
print('Лучшие параметры:', lr_cv.best_params_)
print('Лучшая метрика recall:', lr_cv.best_score_)

Fitting 5 folds for each of 45 candidates, totalling 225 fits
Baseline (Classification):
Лучшие параметры: {'logreg__C': 1, 'logreg__l1_ratio': 1, 'logreg__max_iter': 1000, 'logreg__solver': 'saga'}
Лучшая метрика recall: 0.884031746031746


In [64]:
y_pred = cross_val_predict(lr_cv.best_estimator_, X_val, y_val, cv=cv)
print(classification_report(y_val, y_pred))

              precision    recall  f1-score   support

           0       0.95      0.98      0.97      1035
           1       0.94      0.87      0.90       374

    accuracy                           0.95      1409
   macro avg       0.95      0.93      0.94      1409
weighted avg       0.95      0.95      0.95      1409



## 3. Random Forest

В качестве baseline-модели возьмём случайный лес. Он не требует масштабирования, хорошо работает с дисбалансом классов и устойчив к выбросам.

In [65]:

rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('RF', RandomForestClassifier(random_state=42))
])

rf_param_grid = {
    'RF__n_estimators': [50, 100],  
    'RF__max_depth': [None, 10, 20],
    'RF__min_samples_split': [2, 5],  
    'RF__min_samples_leaf': [1, 4],  
    'RF__max_features': ['sqrt', 'log2'], 
    'RF__class_weight': [None, 'balanced']
}

rf_cv = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_param_grid,
    cv=cv,
    scoring='recall',
    n_jobs=-1
)

rf_cv.fit(X_train, y_train)
print('Random Forest:')
print('Лучшие параметры:', rf_cv.best_params_)
print('Лучшая метрика recall:', rf_cv.best_score_)


Random Forest:
Лучшие параметры: {'RF__class_weight': 'balanced', 'RF__max_depth': 10, 'RF__max_features': 'log2', 'RF__min_samples_leaf': 1, 'RF__min_samples_split': 5, 'RF__n_estimators': 50}
Лучшая метрика recall: 0.9179404761904761


In [66]:
y_pred = cross_val_predict(rf_cv.best_estimator_, X_val, y_val, cv=cv)
print(classification_report(y_val, y_pred))

              precision    recall  f1-score   support

           0       0.96      0.98      0.97      1035
           1       0.94      0.89      0.91       374

    accuracy                           0.96      1409
   macro avg       0.95      0.93      0.94      1409
weighted avg       0.96      0.96      0.95      1409



## 4. Catboost

In [ ]:
catboost_model = CatBoostClassifier(
        cat_features=tuple(categorical),
        random_seed=42,
        verbose=0,
        thread_count=-1,
        auto_class_weights='Balanced',
        early_stopping_rounds=50,
        eval_metric='Recall'
    )

catboost_param_grid = {
    'iterations': [100, 200, 300, 500],
    'depth': [4, 6, 8],
    'learning_rate': [0.01, 0.03, 0.05, 0.07, 0.1],
    'l2_leaf_reg': [1, 3, 5, 7, 9],
    'border_count': [128, 255], 
    'subsample': [0.7, 0.8, 0.9, 1.0]
}

catboost_cv = RandomizedSearchCV(
    catboost_model,
    param_distributions=catboost_param_grid,
    n_iter=30,
    cv=cv,
    scoring='recall',
    verbose=1,
    n_jobs=-1,  # параллелизация по комбинациям параметров
    random_state=42
)

catboost_model.fit(X_train, y_train, 
                      eval_set=(X_val, y_val))
y_pred = catboost_model.predict(X_val)
catboost_recall = recall_score(y_val, y_pred)

print("CatBoost:")
# print(f"Лучшие параметры: {catboost_pipeline.best_params_}")
print(f"Лучшая метрика recall: {catboost_recall:.4f}")

CatBoost:
Лучшая метрика recall: 0.9519


In [57]:
y_pred = cross_val_predict(catboost_model, X_val, y_val, cv=cv)
print(classification_report(y_val, y_pred))

              precision    recall  f1-score   support

           0       0.97      0.97      0.97      1035
           1       0.92      0.91      0.91       374

    accuracy                           0.96      1409
   macro avg       0.95      0.94      0.94      1409
weighted avg       0.96      0.96      0.96      1409

